In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

titles = ["Your Name", "Attack on Titan", "Spirited Away", "Death Note"]
descriptions = [
    "a heartwarming romantic story about two teenagers swapping bodies and falling in love",
    "a dark intense action series about humanity fighting giant monsters and survival",
    "a magical adventure about a girl trapped in a spirit world full of wonder",
    "a dark psychological thriller about a genius using a notebook to kill criminals",
]

query = "something dark and intense"

vectorizer = TfidfVectorizer()
all_text = descriptions + [query]
tfidf_matrix = vectorizer.fit_transform(all_text)

query_vec = tfidf_matrix[-1]
doc_vec = tfidf_matrix[:-1]
scores = cosine_similarity(query_vec, doc_vec)[0] #[0] for 1D


ranked = sorted(zip(titles, scores), key = lambda x : x[1] , reverse= True) #x[1] so that not only string, but scores get displayed as well
for title, score in ranked:
    print(f"{title} : {score:.4f} ")

Attack on Titan : 0.3257 
Death Note : 0.0949 
Your Name : 0.0871 
Spirited Away : 0.0000 


In [2]:
import numpy as np
import pandas as pd

movies = pd.read_csv(r'C:\Users\hp\MoodWatch\data\tmdb_5000_movies.csv')
anime = pd.read_csv(r'C:\Users\hp\MoodWatch\data\mal_anime.csv')

print(movies.columns)
print(anime.columns)
print(movies.head())
print(anime.head())

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='str')
Index(['myanimelist_id', 'title', 'description', 'image', 'Type', 'Episodes',
       'Status', 'Premiered', 'Released_Season', 'Released_Year', 'Source',
       'Genres', 'Themes', 'Studios', 'Producers', 'Demographic', 'Duration',
       'Rating', 'Score', 'Ranked', 'Popularity', 'Members', 'Favorites',
       'characters', 'source_url'],
      dtype='str')
      budget                                             genres  \
0  237000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  300000000  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2  245000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3  250000000  [{"id": 28, "name": "

In [4]:
import sys
sys.path.append('..')

from src.preprocessing import clean_text

movies['cleaned_overview'] = movies['overview'].fillna('').apply(clean_text)
anime['cleaned_description'] = anime['description'].fillna('').apply(clean_text)

print(movies[['title' , 'cleaned_overview']].head())
print(anime[['title', 'cleaned_description']].head())

                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   
3                     The Dark Knight Rises   
4                               John Carter   

                                    cleaned_overview  
0  in the 22nd century a paraplegic marine is dis...  
1  captain barbossa long believed to be dead has ...  
2  a cryptic message from bonds past sends him on...  
3  following the death of district attorney harve...  
4  john carter is a warweary former military capt...  
                             title  \
0                     Cowboy Bebop   
1  Cowboy Bebop: Tengoku no Tobira   
2                           Trigun   
3               Witch Hunter Robin   
4                   Bouken Ou Beet   

                                 cleaned_description  
0  crime is timeless by the year 2071 humanity ha...  
1  another day another bountysuch is the life of

In [5]:
movies['cleaned_overview'].str.split()

0       [in, the, 22nd, century, a, paraplegic, marine...
1       [captain, barbossa, long, believed, to, be, de...
2       [a, cryptic, message, from, bonds, past, sends...
3       [following, the, death, of, district, attorney...
4       [john, carter, is, a, warweary, former, milita...
                              ...                        
4798    [el, mariachi, just, wants, to, play, his, gui...
4799    [a, newlywed, couples, honeymoon, is, upended,...
4800    [signed, sealed, delivered, introduces, a, ded...
4801    [when, ambitious, new, york, attorney, sam, is...
4802    [ever, since, the, second, grade, when, he, fi...
Name: cleaned_overview, Length: 4803, dtype: object

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def remove_stopwords(tokens):
    return [word for word in tokens if word not in ENGLISH_STOP_WORDS]

movies['tokens'] = movies['cleaned_overview'].str.split().apply(remove_stopwords) 

print(movies['tokens'].head())




0    [22nd, century, paraplegic, marine, dispatched...
1    [captain, barbossa, long, believed, dead, come...
2    [cryptic, message, bonds, past, sends, trail, ...
3    [following, death, district, attorney, harvey,...
4    [john, carter, warweary, military, captain, wh...
Name: tokens, dtype: object


In [8]:
inverted_index = {}

for doc_id, tokens in enumerate(movies['tokens']):

    word_counts = {}
    for word in tokens:
        word_counts[word] = word_counts.get(word, 0) + 1

    for word, count in word_counts.items():
        if word not in inverted_index:
            inverted_index[word] = []
        inverted_index[word].append([doc_id, count])


In [9]:
print("Vocabulary size:", len(inverted_index))
print("love:", inverted_index.get('love', 'not found'))

Vocabulary size: 23323
love: [[5, 1], [17, 1], [47, 1], [48, 1], [83, 2], [86, 1], [110, 1], [124, 2], [199, 1], [211, 1], [217, 1], [226, 1], [230, 1], [258, 1], [304, 1], [363, 1], [371, 1], [384, 1], [391, 1], [407, 1], [410, 1], [411, 1], [413, 1], [423, 1], [426, 1], [438, 1], [448, 1], [459, 1], [460, 2], [463, 1], [491, 1], [492, 1], [515, 1], [534, 1], [535, 2], [556, 1], [610, 3], [612, 1], [617, 1], [618, 1], [625, 1], [638, 1], [704, 1], [709, 1], [733, 1], [746, 1], [765, 1], [793, 2], [806, 2], [809, 2], [812, 1], [813, 1], [814, 1], [815, 1], [822, 1], [852, 2], [860, 1], [868, 1], [875, 1], [882, 1], [887, 1], [893, 1], [895, 1], [898, 1], [910, 1], [912, 1], [1023, 1], [1030, 1], [1035, 1], [1061, 1], [1080, 1], [1097, 1], [1098, 2], [1109, 1], [1115, 1], [1117, 1], [1120, 1], [1125, 1], [1132, 1], [1180, 1], [1199, 1], [1204, 1], [1212, 1], [1222, 1], [1235, 1], [1240, 1], [1243, 1], [1260, 1], [1277, 1], [1291, 1], [1300, 1], [1308, 1], [1335, 1], [1337, 1], [1348, 1]